# 03. Registro entre frames consecutivos

Este notebook introduce el primer análisis directamente relacionado con la pregunta de interés para SLAM.

Planteamiento:
- ya se dispone de una pseudo-LiDAR para cada frame,
- por tanto, puede estudiarse si las nubes en `t` y `t+1` presentan una consistencia geométrica suficiente como para alinearse correctamente,
- y si el movimiento estimado a partir de ese registro se aproxima al movimiento real proporcionado por `nuScenes`.

Este procedimiento no constituye un sistema SLAM completo, pero sí proporciona una **prueba razonable de utilidad para tareas de odometría y registro temporal**.


In [1]:
from pathlib import Path
import json
import subprocess


In [2]:
ROOT = Path('/home/clara/ml-depth-pro/slam_readiness_nuscenes')
RUN_SUMMARY = ROOT / 'outputs' / 'scene-0061' / 'run_summary.json'
SCRIPT_PATH = ROOT / 'scripts' / 'evaluate_pairwise_registration.py'

run_summary = json.loads(RUN_SUMMARY.read_text())
run_summary


{'scene_name': 'scene-0061',
 'manifest_path': '/home/clara/ml-depth-pro/slam_readiness_nuscenes/manifests/scene-0061_first5.json',
 'num_processed': 5,
 'samples': [{'sample_token': 'ca9a282c9e77460f8360f564131a8af5',
   'index': 0,
   'timestamp_s': 1532402927.647951,
   'ring_path': '/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/ca9a282c9e77460f8360f564131a8af5/pcd_ring_6cams_ego.ply',
   'pseudo_path': '/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/ca9a282c9e77460f8360f564131a8af5/pcd_pseudolidar_ego.ply',
   'lidar_path': '/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/ca9a282c9e77460f8360f564131a8af5/pcd_lidar_top_ego.ply',
   'ring_num_points': 281010,
   'pseudo_num_points': 23534},
  {'sample_token': '39586f9d59004284a7114a68825e8eec',
   'index': 1,
   'timestamp_s': 1532402928.147847,
   'ring_path': '/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/39586f9d59004284a7114a68825e8eec/pcd_ring_6

## Qué se evalúa en este paso

Para cada pareja consecutiva (`t`, `t+1`) se realizan las siguientes operaciones:
1. cargar la pseudo-LiDAR de ambos frames,
2. intentar alinearlas mediante ICP,
3. estimar el movimiento relativo entre ellas,
4. obtener el movimiento real a partir de la pose de `nuScenes`,
5. y comparar ambos resultados.

Si la pseudo-LiDAR presentara una calidad adecuada para tareas tipo SLAM, cabría esperar:
- un `fitness` elevado en ICP,
- un error de traslación reducido,
- y un error de rotación reducido.


In [3]:
cmd = [
    'python',
    str(SCRIPT_PATH),
    '--run-summary', str(RUN_SUMMARY),
    '--dataroot', '/home/clara/datasets/nuscenes',
    '--version', 'v1.0-mini',
]
print(' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
print('returncode:', result.returncode)


python /home/clara/ml-depth-pro/slam_readiness_nuscenes/scripts/evaluate_pairwise_registration.py --run-summary /home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/run_summary.json --dataroot /home/clara/datasets/nuscenes --version v1.0-mini
/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/pairwise_registration_metrics.json
{
  "scene_name": "scene-0061",
  "num_pairs": 4,
  "pairs": [
    {
      "pair_index": 0,
      "source_token": "ca9a282c9e77460f8360f564131a8af5",
      "target_token": "39586f9d59004284a7114a68825e8eec",
      "dt_s": 0.4998960494995117,
      "gt_relative_translation_m": 4.493198374223651,
      "gt_relative_rotation_deg": 0.41686372685380557,
      "pseudo_icp_fitness": 0.7106527267589715,
      "pseudo_icp_rmse": 0.7483844151631609,
      "pseudo_est_translation_m": 4.906395586625981,
      "pseudo_est_rotation_deg": 0.8138763328437236,
      "pseudo_translation_error_m": 0.9947175184967076,
      "pseudo_rotation_error_deg"

In [4]:
metrics_path = ROOT / 'outputs' / 'scene-0061' / 'pairwise_registration_metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    metrics
else:
    print('Todavia no existe pairwise_registration_metrics.json. Ejecuta antes la celda del script.')


## Interpretación de los resultados

- `pseudo_icp_fitness`: fracción de la nube para la que se encuentran correspondencias razonables.
- `pseudo_icp_rmse`: error medio de dichas correspondencias.
- `pseudo_translation_error_m`: desviación del movimiento estimado respecto al movimiento real, medida en metros.
- `pseudo_rotation_error_deg`: desviación angular respecto a la rotación real.

La comparación con las métricas `gt_*` permite situar el comportamiento de la pseudo-LiDAR con respecto al del LiDAR real del dataset.
